# Niño indices — data exploration
Source file: `data/ENSO_NINO_data.xlsx` (sheet `NINO_Data`), re-created by `data/collect_data.py` from the NOAA CPC `sstoi.indices` file.
Goal: understand the file, check its quality and compare it with the ONI events file. Nothing is dropped or changed in the data here.

In [1]:
# Core libraries for cleaning and exploring data
import pandas as pd
import numpy as np
import pyarrow
# Read the Excel sheet into a DataFrame.
df_raw = pd.read_excel("../data/ENSO_NINO_data.xlsx", sheet_name="NINO_Data")

### loading the data
- The file is an **Excel workbook**, so `pd.read_excel` is the reader, and the sheet is named explicitly (`NINO_Data`).
- It loads **538 rows x 10 columns**: one row = one month, with the sea surface temperature (SST) and its anomaly for four Niño regions.

In [2]:
#copy the data frame to a new variable to avoid changing the original data frame
df = df_raw.copy()
df.head()

,YR,MON,NINO1+2,NINO1+2_ANOM,NINO3,NINO3_ANOM,NINO4,NINO4_ANOM,NINO3.4,NINO3.4_ANOM
0,1982,1,24.28,-0.24,25.84,0.17,28.01,-0.21,26.65,0.08
1,1982,2,25.38,-0.72,26.26,-0.11,27.99,-0.11,26.54,-0.20
2,1982,3,25.22,-1.38,26.92,-0.25,28.18,-0.05,27.09,-0.14
3,1982,4,24.57,-1.16,27.52,-0.05,28.61,0.10,27.83,0.02
4,1982,5,24.00,-0.62,27.70,0.49,29.19,0.40,28.37,0.49


### first look at the rows
- The series starts in **1982-01** (the collection script keeps only years from 1982).
- Every region has two columns: the SST in °C and its anomaly, e.g. Niño 1+2 is `24.28` °C with an anomaly of `-0.24` in January 1982.
- There is **no date column**: the month is split into `YR` and `MON`.

In [3]:
print("Shape:", df.shape)
df.tail(3)

Shape: (538, 10)


,YR,MON,NINO1+2,NINO1+2_ANOM,NINO3,NINO3_ANOM,NINO4,NINO4_ANOM,NINO3.4,NINO3.4_ANOM
535,2026,8,24.93,4.08,28.35,3.13,29.63,0.93,29.42,2.52
536,2026,9,25.27,4.69,28.65,3.64,29.62,0.95,29.60,2.84
537,2026,10,26.12,5.25,29.24,4.16,29.88,1.20,30.01,3.25


### size and ordering of the file
- Shape confirmed: **(538, 10)**.
- The last row is **2026-10**: Niño 1+2 anomaly `+5.25`, Niño 3 `+4.16`, Niño 3.4 `+3.25`.
- The README says the file ends in 2026-08, so the file now holds **two more months** than the README states.

In [4]:
# Column names
print(f"Column names: {df.columns.tolist()}")

# Which sheets does the workbook have?
print(pd.ExcelFile("../data/ENSO_NINO_data.xlsx").sheet_names)

Column names: ['YR', 'MON', 'NINO1+2', 'NINO1+2_ANOM', 'NINO3', 'NINO3_ANOM', 'NINO4', 'NINO4_ANOM', 'NINO3.4', 'NINO3.4_ANOM']
['NINO_Data']


### column names
| Column | Meaning |
|---|---|
| `YR`, `MON` | Year and month (1 to 12) |
| `NINO1+2`, `NINO1+2_ANOM` | SST and anomaly (°C), region closest to the South American coast |
| `NINO3`, `NINO3_ANOM` | SST and anomaly (°C), eastern-central Pacific |
| `NINO3.4`, `NINO3.4_ANOM` | SST and anomaly (°C), central Pacific (the region the ONI is based on) |
| `NINO4`, `NINO4_ANOM` | SST and anomaly (°C), western-central Pacific |

- The workbook has a single sheet, `NINO_Data`.

In [5]:
df_raw.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 538 entries, 0 to 537
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   YR            538 non-null    int64  
 1   MON           538 non-null    int64  
 2   NINO1+2       538 non-null    float64
 3   NINO1+2_ANOM  538 non-null    float64
 4   NINO3         538 non-null    float64
 5   NINO3_ANOM    538 non-null    float64
 6   NINO4         538 non-null    float64
 7   NINO4_ANOM    538 non-null    float64
 8   NINO3.4       538 non-null    float64
 9   NINO3.4_ANOM  538 non-null    float64
dtypes: float64(8), int64(2)
memory usage: 42.2 KB


### data types and memory
- 2 `int64` columns (`YR`, `MON`) and 8 `float64` columns.
- All 10 columns show **538 non-null** values.
- Memory is about **42 KB**: this file is tiny, so memory is not a concern.

In [6]:
# Null counts and duplicates
print("Total nulls   :", df_raw.isna().sum().sum())
print("Duplicate rows:", df_raw.duplicated().sum())
print("Duplicate YR + MON pairs:", df_raw.duplicated(subset=["YR", "MON"]).sum())

Total nulls   : 0
Duplicate rows: 0
Duplicate YR + MON pairs: 0


### missing values and duplicates
- **0 missing values** in the whole sheet.
- **No duplicate rows and no duplicate `YR` + `MON` pairs**, so each month appears exactly once.

In [7]:
# 1. YR and MON fit in int16, 2. build a real date column from them
for col in ["YR", "MON"]:
    df[col] = df[col].astype("int16")
df["date"] = pd.to_datetime(dict(year=df["YR"], month=df["MON"], day=1))

print(df[["YR", "MON", "date"]].dtypes)
print("Before:", round(df_raw.memory_usage(deep=True).sum() / 1024, 1), "KB")
print("After :", round(df.drop(columns="date").memory_usage(deep=True).sum() / 1024, 1), "KB (without the new date column)")

YR               int16
MON              int16
date    datetime64[us]
dtype: object
Before: 42.2 KB
After : 35.9 KB (without the new date column)


### fixing the data types
- `YR` and `MON` fit in `int16`, which cuts the memory of the data columns from **42.2 KB to 35.9 KB** (about 15% less).
- A real `date` column (first day of the month) is built from `YR` and `MON`. This is the column that will be used to join with the other files.

In [8]:
# Is there exactly one row per month, with no gaps?
expected = pd.date_range(df["date"].min(), df["date"].max(), freq="MS")
print("Range   :", df["date"].min().date(), "->", df["date"].max().date())
print("Expected:", len(expected), " Found:", len(df))
print("Missing months:", expected.difference(df["date"]).tolist())

# Months per year (only the first and the last year should be incomplete)
per_year = df.groupby("YR").size()
print(per_year.value_counts())
print(per_year.tail(2))

Range   : 1982-01-01 -> 2026-10-01
Expected: 538  Found: 538
Missing months: []
12    44
10     1
Name: count, dtype: int64
YR
2025    12
2026    10
dtype: int64


### is the series complete?
- **538 months from 1982-01 to 2026-10, with no gaps.**
- 44 years (1982 to 2025) have 12 rows and **2026 has 10** (January to October), so 2026 is a partial year.

In [9]:
# Summary stats for the numeric columns
df.drop(columns=["date"]).describe().round(2)

,YR,MON,NINO1+2,NINO1+2_ANOM,NINO3,NINO3_ANOM,NINO4,NINO4_ANOM,NINO3.4,NINO3.4_ANOM
count,538.00,538.00,538.00,538.00,538.00,538.00,538.00,538.00,538.00,538.00
mean,2003.92,6.48,23.32,0.03,26.02,-0.01,28.50,-0.06,27.06,-0.03
std,12.95,3.45,2.34,1.13,1.26,0.91,0.70,0.65,0.98,0.87
min,1982.00,1.00,19.06,-1.90,23.38,-2.16,26.36,-1.87,24.56,-2.22
25%,1993.00,3.25,21.26,-0.71,25.00,-0.63,28.02,-0.54,26.34,-0.64
50%,2004.00,6.00,23.28,-0.18,26.00,-0.12,28.58,0.01,27.08,-0.09
75%,2015.00,9.00,25.35,0.52,27.04,0.48,29.01,0.40,27.74,0.47
max,2026.00,12.00,28.51,5.25,29.24,4.16,30.22,1.55,30.01,3.25


### numeric summary
- **SST ranges:** Niño 1+2 `19.06` to `28.51` °C (the most variable region, std `2.34`), Niño 3 `23.38` to `29.24`, Niño 3.4 `24.56` to `30.01`, Niño 4 `26.36` to `30.22` (the warmest and steadiest, std `0.70`).
- **Anomaly ranges:** Niño 1+2 `-1.90` to `+5.25`, Niño 3 `-2.16` to `+4.16`, Niño 3.4 `-2.22` to `+3.25`, Niño 4 `-1.87` to `+1.55`.
- The anomaly means are all close to zero (`-0.06` to `+0.03`), as expected for anomalies.

In [10]:
# Do the SST columns and their anomalies agree? (SST - anomaly = climatology of that month)
regions = ["NINO1+2", "NINO3", "NINO4", "NINO3.4"]
clim = pd.DataFrame({r: df[r] - df[f"{r}_ANOM"] for r in regions})
clim["MON"] = df["MON"]
spread = clim.groupby("MON")[regions].std().max().round(3)
print("Largest spread of the implied climatology within one calendar month:")
print(spread)

# Mean implied climatology by calendar month (what a 'normal' month looks like)
clim.groupby("MON")[regions].mean().round(2)

Largest spread of the implied climatology within one calendar month:
NINO1+2    0.005
NINO3      0.005
NINO4      0.005
NINO3.4    0.005
dtype: float64


,NINO1+2,NINO3,NINO4,NINO3.4
MON,,,,
1,24.52,25.67,28.22,26.56
2,26.10,26.38,28.10,26.73
3,26.60,27.16,28.23,27.23
4,25.73,27.57,28.51,27.81
5,24.61,27.21,28.79,27.88
6,23.18,26.64,28.84,27.71
7,21.83,25.87,28.79,27.30
8,20.86,25.22,28.69,26.90
9,20.58,25.01,28.68,26.76


### the climatology behind the anomalies
- `SST - anomaly` is **the same for every year within a calendar month** (largest spread `0.005`, which is rounding). So the anomalies are measured against **one fixed normal per calendar month**.
- This is different from the events file, where `ClimAdjust` drifts over the years.
- The normal cycle differs a lot by region: Niño 1+2 swings from `20.58` °C (September) to `26.60` °C (March), Niño 3.4 only from `26.56` to `27.88` °C, and Niño 4 from `28.10` to `28.84` °C.

In [11]:
# Warm / cool months using the usual +-0.5 degC threshold on the Nino 3.4 anomaly
a = df["NINO3.4_ANOM"]
print("Warm months (>= +0.5):", (a >= 0.5).sum(), " Cool months (<= -0.5):", (a <= -0.5).sum(), " In between:", a.between(-0.5, 0.5, inclusive="neither").sum())

# Strongest monthly anomalies in each region
for r in regions:
    top = df.loc[df[f"{r}_ANOM"].idxmax()]
    low = df.loc[df[f"{r}_ANOM"].idxmin()]
    print(f"{r:8s} max {top[f'{r}_ANOM']:+.2f} ({top['date']:%Y-%m})   min {low[f'{r}_ANOM']:+.2f} ({low['date']:%Y-%m})")

Warm months (>= +0.5): 129  Cool months (<= -0.5): 175  In between: 234
NINO1+2  max +5.25 (2026-10)   min -1.90 (2007-11)
NINO3    max +4.16 (2026-10)   min -2.16 (1988-06)
NINO4    max +1.55 (2015-11)   min -1.87 (1988-11)
NINO3.4  max +3.25 (2026-10)   min -2.22 (1988-11)


### warm months, cool months and extremes
- On the Niño 3.4 anomaly with the usual +-0.5 °C threshold there are **129 warm months, 175 cool months and 234 in between**. This is a *monthly* threshold; an ONI event also needs a 3-month mean over 5 months.
- **The most recent month is the largest anomaly in the file for three of the four regions**: Niño 1+2 (`+5.25`), Niño 3 (`+4.16`) and Niño 3.4 (`+3.25`) all peak in **2026-10**. Niño 4 peaked earlier (`+1.55` in 2015-11).
- The coldest months are older: Niño 3.4 `-2.22` in 1988-11, Niño 3 `-2.16` in 1988-06, Niño 1+2 `-1.90` in 2007-11.

In [12]:
# How the current year compares with the strongest past years (Nino 3.4 anomaly, monthly)
piv = df.pivot(index="MON", columns="YR", values="NINO3.4_ANOM")
piv[[1982, 1997, 2015, 2023, 2026]]

YR,1982,1997,2015,2023,2026
MON,,,,,
1,0.08,-0.61,0.51,-0.69,-0.54
2,-0.20,-0.39,0.75,-0.44,-0.20
3,-0.14,-0.36,0.44,-0.01,0.03
4,0.02,-0.11,0.82,0.19,0.47
5,0.49,0.40,0.83,0.47,0.94
6,0.65,0.81,1.02,0.88,1.55
7,0.27,1.27,1.26,1.07,2.03
8,0.86,1.68,1.65,1.30,2.52
9,1.24,1.84,1.79,1.53,2.84


### 2026 against the strongest past years (Niño 3.4 anomaly)
- From **May to October 2026 the anomaly is above all four comparison years** (1982, 1997, 2015, 2023). In October it is `+3.25` against `+2.21` (2015), `+1.96` (1997), `+1.73` (1982) and `+1.59` (2023).
- The earlier events peaked in **November or December** (`+2.72` in 2015-11, `+2.21` in 1982-12, `+2.11` in 1997-11, `+1.99` in 2023-12), so 2026 still has two months to go in this file.
- January 2026 started cool (`-0.54`), like 1997 and 2023 (`-0.61` and `-0.69`).

In [13]:
# How closely do the four regions move together? (anomaly correlations)
df[[f"{r}_ANOM" for r in regions]].corr().round(2)

,NINO1+2_ANOM,NINO3_ANOM,NINO4_ANOM,NINO3.4_ANOM
NINO1+2_ANOM,1.00,0.83,0.47,0.67
NINO3_ANOM,0.83,1.00,0.74,0.94
NINO4_ANOM,0.47,0.74,1.00,0.88
NINO3.4_ANOM,0.67,0.94,0.88,1.00


### how the four regions move together
- Niño 3 and Niño 3.4 are almost the same signal (**correlation 0.94**); Niño 3.4 and Niño 4 are also close (**0.88**).
- **Niño 1+2 is the most independent** (0.47 with Niño 4, 0.67 with Niño 3.4). It captures the coastal signal, which the README flags as relevant for Peru.

In [14]:
# Cross-check with the events file: same Nino 3.4 anomaly?
ev = pd.read_csv("../data/el-nino-southern-oscillation-enso-el-nino-and-la-nina-events.csv", parse_dates=["date"])
both = df.merge(ev[["date", "ANOM"]], on="date", how="inner")
diff = (both["NINO3.4_ANOM"] - both["ANOM"])
print("Months in common:", len(both), " First:", both["date"].min().date(), " Last:", both["date"].max().date())
print("Correlation:", round(both["NINO3.4_ANOM"].corr(both["ANOM"]), 4))
print("Difference  mean:", round(diff.mean(), 3), " max |diff|:", round(diff.abs().max(), 2))
print("Months only in the Nino file:", sorted(set(df["date"]) - set(ev["date"]))[:1], "...", len(set(df["date"]) - set(ev["date"])))

Months in common: 534  First: 1982-01-01  Last: 2026-06-01
Correlation: 0.98
Difference  mean: -0.096  max |diff|: 0.66
Months only in the Nino file: [Timestamp('2026-07-01 00:00:00')] ... 4


### cross-check with the events file
- The two files share **534 months** (1982-01 to 2026-06) and the Niño 3.4 anomaly is almost the same series (**correlation 0.98**).
- They are **not identical**: on average the Niño file is `0.096` °C lower, and a single month can differ by up to `0.66` °C. This fits two different baselines (fixed normals here, a moving `ClimAdjust` in the events file).
- **4 months exist only in this file** (2026-07 to 2026-10), because the events file stops in 2026-06.

In [15]:
# Save an untouched Parquet copy of the raw sheet (checkpoint before any cleaning)
import os
os.makedirs("../data/parquet", exist_ok=True)
df_raw.to_parquet("../data/parquet/nino_indices_raw.parquet")

back = pd.read_parquet("../data/parquet/nino_indices_raw.parquet")
print("Round trip identical:", back.equals(df_raw))
print("Excel  :", round(os.path.getsize("../data/ENSO_NINO_data.xlsx") / 1024, 1), "KB")
print("Parquet:", round(os.path.getsize("../data/parquet/nino_indices_raw.parquet") / 1024, 1), "KB")

Round trip identical: True
Excel  : 30.3 KB
Parquet: 21.3 KB


### Parquet checkpoint
- The raw sheet is saved as `data/parquet/nino_indices_raw.parquet`, and reading it back returns an **identical** DataFrame.
- Size drops from **30.3 KB (Excel) to 21.3 KB**. This is the untouched copy; cleaning is done later in SQL.

### Conclusion: coverage and data quality

| Check | Result |
|---|---|
| Rows x columns | 538 x 10 |
| Period | 1982-01 to 2026-10, monthly (2026 has 10 months) |
| Gaps / duplicates / missing values | none / none / none |
| Regions | Niño 1+2, 3, 4 and 3.4 (SST and anomaly each) |
| Agreement with the events file | correlation 0.98 over 534 months |

### Conclusion: decisions for the cleaning step

| Topic | Action |
|---|---|
| `YR` + `MON` | Combine into one `date` (done here) |
| Region columns | Keep all four, both SST and anomaly (Niño 1+2 adds coastal information that Niño 3.4 does not) |
| Anomalies from two files | **Do not mix** this `NINO3.4_ANOM` with `ANOM` from the events file in one analysis: the baselines differ (mean gap `0.096` °C, up to `0.66` °C) |
| README | Update the end date of this file: it now ends in **2026-10**, not 2026-08 |
| 2026 | Partial year (10 months): do not compare its yearly average with full years |